# Krishi Mitra — Dataset Downloader (Colab)

Downloads required datasets into Google Drive using Kaggle API.

You can authenticate in **either** way:
- Upload `kaggle.json` into Colab, or
- Enter your Kaggle **username** + **API key** (this notebook will generate `kaggle.json`).

Datasets:
1. `atharvaingle/crop-recommendation-dataset`
2. `emmarex/plantdisease`
3. `minhhuy2810/rice-diseases-image-dataset`
4. `akshatgupta7/crop-yield-in-indian-states-dataset`


In [ ]:
!pip -q install kaggle

from google.colab import drive
drive.mount('/content/drive')

MODEL_SAVE_PATH = '/content/drive/MyDrive/krishi_mitra_models/'
DATA_PATH = '/content/drive/MyDrive/krishi_mitra_data/'

import os
from pathlib import Path
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

raw_dir = Path(DATA_PATH) / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)
raw_dir

## 1) Kaggle authentication

### Option A: Upload `kaggle.json`
Upload the file to Colab (left sidebar → Files) and run the cell.

### Option B: Paste username + API key
If you don't want to upload `kaggle.json`, enter your username and key; we will generate the file.


In [ ]:
import json
from pathlib import Path
import getpass

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)

uploaded = Path('kaggle.json')
target = kaggle_dir / 'kaggle.json'

if uploaded.exists():
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json
    print('Using uploaded kaggle.json')
else:
    print('No kaggle.json found in runtime. Creating one from credentials...')
    username = input('Kaggle username: ').strip()
    key = getpass.getpass('Kaggle API key (input hidden): ').strip()
    target.write_text(json.dumps({'username': username, 'key': key}), encoding='utf-8')
    !chmod 600 /root/.kaggle/kaggle.json
    print('Created /root/.kaggle/kaggle.json')

assert target.exists(), 'Kaggle auth file not created'


## 2) Download + unzip all datasets

Files are downloaded to `DATA_PATH/raw/` in your Drive.


In [ ]:
from pathlib import Path
import subprocess

datasets = [
    ('atharvaingle/crop-recommendation-dataset', 'crop-recommendation-dataset.zip'),
    ('emmarex/plantdisease', 'plantdisease.zip'),
    ('minhhuy2810/rice-diseases-image-dataset', 'rice-diseases-image-dataset.zip'),
    ('akshatgupta7/crop-yield-in-indian-states-dataset', 'crop-yield-in-indian-states-dataset.zip'),
]

for dset, zip_name in datasets:
    print('Downloading:', dset)
    !kaggle datasets download -d {dset} -p "{raw_dir}" -q

print('Download complete. Zip files in:', raw_dir)
sorted([p.name for p in raw_dir.glob('*.zip')])[:20]

In [ ]:
import zipfile
from pathlib import Path

def unzip_all(zips_dir: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    for z in zips_dir.glob('*.zip'):
        print('Unzipping:', z.name)
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall(out_dir)

unzipped_dir = raw_dir / 'unzipped'
unzip_all(raw_dir, unzipped_dir)
print('Unzipped to:', unzipped_dir)
len(list(unzipped_dir.rglob('*')))

## 3) Next steps

- Model 1: use `Crop_recommendation.csv` from the crop recommendation dataset.
- Model 2: curate PlantVillage + Rice datasets into `DATA_PATH/processed/disease_images/train|val|test/<class>/...`
- Model 3: use yield CSV(s) and merge with any additional sources.
- Model 4: synthetic (already generated by notebook/scripts).
